# 01 · Explore Universe & Spread Differential

Phase 2 verification for the fallen-angel trade (CNC default).

1. Pull and **screen** the issuer bond universe, then **rank** it on the composite score.
2. Build the **BB+ healthcare peer basket** and basket weights.
3. Compute the **weighted-average Z-spread differential** (target vs peers) and its rolling z-score.
4. Chart the differential with entry/exit thresholds.

Run with `uv run jupyter lab` from the `fallen_angels/` dir so the package and a logged-in Bloomberg Terminal are on the path. No notebook magics are used.

In [ ]:
import logging

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fallen_angels.config import load_config
from fallen_angels import data_pull as dp
from fallen_angels import universe as uni
from fallen_angels import comparables as comp
from fallen_angels import signals as sig

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

ISSUER = "CNC"
cfg = load_config(ISSUER)
ZFIELD = cfg.bloomberg.zspread_field
print(cfg.issuer.name, "| peers:", [p.ticker for p in cfg.peers])

## 1. Universe: pull, screen, rank

`get_bond_universe` resolves the bond chain and static fields; `filter_universe` applies the maturity/size/seniority/currency screen.

In [ ]:
raw = dp.get_bond_universe(ISSUER, config=cfg)
screened = uni.filter_universe(raw, cfg)
print(f"{len(raw)} bonds -> {len(screened)} after screen")
screened.head()

In [ ]:
# Latest Z-spread snapshot for the screened bonds -> spread-vs-curve residual.
securities = screened["security"].tolist()
end = pd.Timestamp.today().normalize()
snap = dp.get_bond_timeseries(
    securities, end - pd.Timedelta(days=10), end, [ZFIELD], config=cfg
)
latest_z = snap.sort_values("date").groupby("security")["value"].last()
screened = screened.assign(z_spread=screened["security"].map(latest_z))

resid = uni.spread_vs_curve(
    screened.dropna(subset=["z_spread"]), spread_col="z_spread"
)
ranked = uni.score_universe(screened, cfg, spread_residual=resid)
ranked[[
    "security", "security_des", "years_to_maturity", "amt_outstanding",
    "z_spread", "score",
]].head(15)

## 2. Peer basket

`build_comparables` screens each peer's bonds the same way and assigns `equal_issuer` weights (each peer equal-weighted, amount-weighted within).

In [ ]:
peers = comp.build_comparables(cfg)
peer_weights = comp.to_security_weights(peers)
peers.groupby("peer").agg(
    bonds=("security", "size"), weight=("weight", "sum")
)

In [ ]:
# Target long basket: top-scored CNC bonds, amount-weighted.
TOP_N = 8
basket = ranked.head(TOP_N).copy()
basket["weight"] = comp.assign_basket_weights(basket, scheme="amount").to_numpy()
cnc_weights = comp.to_security_weights(basket)
cnc_weights

## 3. Spread differential & rolling z-score

Pull ~3y of Z-spread history for both baskets and compute the differential and its 504-day rolling z-score.

In [ ]:
hist_start = end - pd.Timedelta(days=365 * 3)
cnc_ts = dp.get_bond_timeseries(
    list(cnc_weights.index), hist_start, end, [ZFIELD], config=cfg
)
peer_ts = dp.get_bond_timeseries(
    list(peer_weights.index), hist_start, end, [ZFIELD], config=cfg
)
signal = sig.compute_signal(cnc_ts, cnc_weights, peer_ts, peer_weights, cfg)
print(sig.latest_state(signal, cfg))
signal.tail()

## 4. Chart: weighted-average Z-spreads + differential z-score

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=signal["date"], y=signal["target_wavg"], name="CNC w-avg Z (bps)"))
fig.add_trace(go.Scatter(x=signal["date"], y=signal["peer_wavg"], name="Peer w-avg Z (bps)"))
fig.add_trace(
    go.Scatter(x=signal["date"], y=signal["z_score"], name="Differential z-score",
               line=dict(dash="dot", color="firebrick")),
    secondary_y=True,
)
fig.add_hline(y=cfg.thresholds.entry_z, line_dash="dash", line_color="green",
              annotation_text=f"entry +{cfg.thresholds.entry_z}σ", secondary_y=True)
fig.add_hline(y=cfg.thresholds.exit_converge_z, line_dash="dash", line_color="gray",
              annotation_text=f"exit +{cfg.thresholds.exit_converge_z}σ", secondary_y=True)
fig.update_yaxes(title_text="Z-spread (bps)", secondary_y=False)
fig.update_yaxes(title_text="Differential z-score", secondary_y=True)
fig.update_layout(
    title=f"{cfg.issuer.short} vs BB+ healthcare peers — Z-spread & differential z-score",
    height=520, legend=dict(orientation="h", y=1.08),
)
fig.show()